# MonoDGP M58: fixed-shape FP32 Core ML conversion

Run every cell on a Colab GPU with high RAM. This notebook performs no training. It reconstructs the exact M57 source, verifies the reviewed M57 evidence, traces one fixed-shape sample, and attempts an FP32 ML Program conversion. It does not run Core ML prediction or authorize deployment.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
from collections import deque
import hashlib, json, os, shlex, shutil, subprocess, sys
MOBILE_REPO=Path('/content/mobile_adas3d')
MONODGP_REPO=Path('/content/MonoDGP_M58')
MONODGP_COMMIT='aa059a18214aebf644510e7f0793971b403f9d14'
DRIVE_DATASET_ROOT=Path('/content/drive/MyDrive/datasets/kitti')
LOCAL_DATASET_ROOT=Path('/content/kitti')
SPLIT_DIR=Path('/content/drive/MyDrive/mobile_adas3d_splits/kitti_chen')
DATASET_ROOT=Path('/content/monodgp_kitti_m56d')
M57_ROOT=Path('/content/drive/MyDrive/mobile_adas3d_outputs/compression/monodgp_m57_deformable_attention')
M57_MANIFEST=M57_ROOT/'m57_deformable_attention_manifest.json'
M57_SMOKE=M57_ROOT/'m57_deformable_attention_smoke.json'
M57_GATE=M57_ROOT/'complete_evaluation/m57_portable_attention_gate.json'
M57_COMPARISON=M57_ROOT/'complete_evaluation/m57_portable_attention_comparison.csv'
OUTPUT_ROOT=Path('/content/drive/MyDrive/mobile_adas3d_outputs/compression/monodgp_m58_coreml_conversion')
LOG_DIR=OUTPUT_ROOT/'colab_logs'
REPORT=OUTPUT_ROOT/'m58_coreml_export_gate.json'
REVIEWED={
 M57_MANIFEST:'7c4757798dfff4d95053912730faff4c5d7a4626ff41a539ad87b2f9193eb313',
 M57_SMOKE:'2a96cffd8e1e6b77f2c547c2b94dca4bde2e72bf4185d853ee7bca71ed28b3b0',
 M57_GATE:'63f2b71d1f5be69bad31907db229f79e39fb17c033ba43999c6fdfe1df3b97a7',
 M57_COMPARISON:'687b15cd4a044981f4e00fa038f2fc5e7053cee692e13bfdb85fe2f90656412d',
}
def sha256(path):
 digest=hashlib.sha256()
 with Path(path).open('rb') as handle:
  for block in iter(lambda:handle.read(1024*1024),b''): digest.update(block)
 return digest.hexdigest()
def run(command,cwd=None,env=None):
 command=[str(item) for item in command]; print('+',shlex.join(command),flush=True)
 merged=os.environ.copy(); merged.update(env or {})
 result=subprocess.run(command,cwd=cwd,env=merged)
 if result.returncode: raise RuntimeError(f'Exit {result.returncode}: {shlex.join(command)}')
def run_logged(command,cwd,log_path,env=None):
 command=[str(item) for item in command]; print('+',shlex.join(command),flush=True)
 log_path=Path(log_path); log_path.parent.mkdir(parents=True,exist_ok=True)
 merged=os.environ.copy(); merged.update(env or {}); tail=deque(maxlen=120)
 with log_path.open('w',encoding='utf-8') as log:
  process=subprocess.Popen(command,cwd=cwd,env=merged,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,bufsize=1)
  for line in process.stdout:
   print(line,end='',flush=True); log.write(line); tail.append(line.rstrip())
  code=process.wait()
 if code: raise RuntimeError(f'Exit {code}; full log={log_path}\n'+'\n'.join(tail))
 return log_path
OUTPUT_ROOT.mkdir(parents=True,exist_ok=True)
run(['nvidia-smi'])


In [ ]:
# Restore the exact repositories and M57 portable operator.
if not MOBILE_REPO.exists(): run(['git','clone','https://github.com/Ali-RT/mobile_adas3d.git',MOBILE_REPO])
else: run(['git','pull','--ff-only'],cwd=MOBILE_REPO)
if not MONODGP_REPO.exists(): run(['git','clone','https://github.com/PuFanqi23/MonoDGP.git',MONODGP_REPO])
run(['git','fetch','--all'],cwd=MONODGP_REPO)
run(['git','checkout',MONODGP_COMMIT],cwd=MONODGP_REPO)
run([sys.executable,'-m','pip','install','-q','coremltools==9.0','pyyaml','scipy','opencv-python-headless','numba','scikit-image','scikit-learn','tqdm','ninja','pandas'])
run([sys.executable,'scripts/patch_monodgp_colab_compat.py','--monodgp-repo',MONODGP_REPO],cwd=MOBILE_REPO)
run([sys.executable,'scripts/patch_monodgp_m54_training.py','--monodgp-repo',MONODGP_REPO],cwd=MOBILE_REPO)
run([sys.executable,'scripts/patch_monodgp_m57_deformable_attention.py','--monodgp-repo',MONODGP_REPO],cwd=MOBILE_REPO)
ops=MONODGP_REPO/'lib/models/monodgp/ops'
shutil.rmtree(ops/'build',ignore_errors=True)
run([sys.executable,'setup.py','build','install'],cwd=ops,env={'MAX_JOBS':'2'})
run([sys.executable,'-c','import torch, coremltools, MultiScaleDeformableAttention; print(torch.__version__,torch.version.cuda,coremltools.__version__)'],cwd=MONODGP_REPO)


In [ ]:
# Restore the exact dataset view and verify all four reviewed M57 artifacts.
def resolve(root,names):
 for name in names:
  path=root/name
  if path.is_dir(): return path
sources={key:resolve(LOCAL_DATASET_ROOT,names) or resolve(DRIVE_DATASET_ROOT,names) for key,names in {
 'image_2':['training/image_2','training/image_02'],
 'label_2':['training/label_2','training/label_02'],
 'calib':['training/calib'],
}.items()}
if any(path is None for path in sources.values()): raise FileNotFoundError(sources)
(DATASET_ROOT/'training').mkdir(parents=True,exist_ok=True)
(DATASET_ROOT/'ImageSets').mkdir(parents=True,exist_ok=True)
for name,target in sources.items():
 link=DATASET_ROOT/'training'/name
 if link.is_symlink() and link.resolve()==target.resolve(): continue
 if link.exists() or link.is_symlink(): raise RuntimeError(f'Refusing to replace {link}')
 link.symlink_to(target,target_is_directory=True)
for split in ('train','val'): shutil.copy2(SPLIT_DIR/f'{split}.txt',DATASET_ROOT/'ImageSets'/f'{split}.txt')
assert len((DATASET_ROOT/'ImageSets/train.txt').read_text().splitlines())==3712
assert len((DATASET_ROOT/'ImageSets/val.txt').read_text().splitlines())==3769
for path,expected in REVIEWED.items():
 if not path.is_file(): raise FileNotFoundError(path)
 actual=sha256(path)
 if actual != expected: raise RuntimeError(f'Reviewed evidence changed: {path}: {actual}')
print('Exact reviewed M57 evidence found; M58 experimental conversion is authorized.')


In [ ]:
# Trace and convert the exact selected graph. This can be CPU-heavy despite the GPU runtime.
EXPORT_LOG=run_logged([
 sys.executable,'-u','scripts/export_monodgp_m58_coreml.py',
 '--monodgp-repo',MONODGP_REPO,
 '--m57-manifest',M57_MANIFEST,'--m57-smoke',M57_SMOKE,
 '--m57-gate',M57_GATE,'--m57-comparison',M57_COMPARISON,
 '--dataset-root',DATASET_ROOT,'--split-dir',SPLIT_DIR,
 '--output-dir',OUTPUT_ROOT,
],MOBILE_REPO,LOG_DIR/'m58_coreml_export.log',env={'MONODGP_PORTABLE_DEFORM_ATTN':'1'})
report=json.loads(REPORT.read_text())
if not report.get('complete') or not report.get('all_export_gates_passed'): raise RuntimeError(report)
if not report.get('macos_coreml_prediction_parity_authorized'): raise RuntimeError(report)
if report.get('physical_device_gate_authorized') or report.get('deployment_authorized') or report.get('product_safety_qualified'): raise RuntimeError(report)
print(json.dumps(report,indent=2))
print('STOP: return',REPORT)


## Stop point 1

Return `m58_coreml_export_gate.json`. Keep the generated `.mlpackage`, ZIP, TorchScript, and reference-I/O files in Google Drive. Do not attempt iPhone deployment. A pass authorizes a separately reviewed macOS Core ML prediction-parity step only.
